## Sample E-Commerce Database

This notebook contains three interconnected tables representing a simple e-commerce system:

### Tables Structure

**1. customers** (8 records)
* `customer_id` - Unique identifier
* `name` - Customer full name
* `email` - Contact email
* `city` - Location
* `signup_date` - Registration date
* `membership_tier` - Premium, Standard, or Basic

**2. products** (8 records)
* `product_id` - Unique identifier
* `product_name` - Product name
* `category` - Electronics, Furniture, or Office Supplies
* `price` - Unit price
* `stock_quantity` - Available inventory

**3. orders** (12 records)
* `order_id` - Unique identifier
* `customer_id` - Links to customers table
* `product_id` - Links to products table
* `quantity` - Units ordered
* `order_date` - Purchase date
* `total_amount` - Total order value

### Relationships
* Orders → Customers (many-to-one via `customer_id`)
* Orders → Products (many-to-one via `product_id`)

In [0]:
%sql
-- Create a customers table with sample data
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@email.com', 'New York', '2023-01-15', 'Premium'),
  (2, 'Bob Smith', 'bob@email.com', 'Los Angeles', '2023-02-20', 'Standard'),
  (3, 'Carol White', 'carol@email.com', 'Chicago', '2023-03-10', 'Premium'),
  (4, 'David Brown', 'david@email.com', 'Houston', '2023-04-05', 'Basic'),
  (5, 'Emma Davis', 'emma@email.com', 'Phoenix', '2023-05-12', 'Standard'),
  (6, 'Frank Wilson', 'frank@email.com', 'New York', '2023-06-18', 'Premium'),
  (7, 'Grace Lee', 'grace@email.com', 'San Francisco', '2023-07-22', 'Standard'),
  (8, 'Henry Martinez', 'henry@email.com', 'Seattle', '2023-08-30', 'Basic')
AS customers(customer_id, name, email, city, signup_date, membership_tier);

SELECT * FROM customers;

In [0]:
%sql
-- Create a products table with sample data
CREATE OR REPLACE TEMP VIEW products AS
SELECT * FROM VALUES
  (101, 'Laptop', 'Electronics', 1200.00, 45),
  (102, 'Wireless Mouse', 'Electronics', 25.99, 150),
  (103, 'Desk Chair', 'Furniture', 299.99, 30),
  (104, 'Monitor 27"', 'Electronics', 350.00, 60),
  (105, 'Keyboard', 'Electronics', 79.99, 120),
  (106, 'Standing Desk', 'Furniture', 599.99, 20),
  (107, 'Webcam HD', 'Electronics', 89.99, 80),
  (108, 'Notebook Set', 'Office Supplies', 15.99, 200)
AS products(product_id, product_name, category, price, stock_quantity);

SELECT * FROM products;

In [0]:
%sql
-- Create an orders table with sample data
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1001, 1, 101, 2, '2024-01-15', 2400.00),
  (1002, 2, 102, 1, '2024-01-16', 25.99),
  (1003, 1, 105, 1, '2024-01-18', 79.99),
  (1004, 3, 103, 1, '2024-01-20', 299.99),
  (1005, 4, 108, 5, '2024-01-22', 79.95),
  (1006, 5, 104, 1, '2024-01-25', 350.00),
  (1007, 3, 101, 1, '2024-02-01', 1200.00),
  (1008, 6, 107, 2, '2024-02-05', 179.98),
  (1009, 2, 106, 1, '2024-02-10', 599.99),
  (1010, 7, 102, 3, '2024-02-15', 77.97),
  (1011, 8, 105, 1, '2024-02-20', 79.99),
  (1012, 1, 104, 2, '2024-03-01', 700.00)
AS orders(order_id, customer_id, product_id, quantity, order_date, total_amount);

SELECT * FROM orders;

In [0]:
%sql
-- Query: Total spending by customer
SELECT 
  c.customer_id,
  c.name,
  c.city,
  c.membership_tier,
  COUNT(o.order_id) as total_orders,
  SUM(o.total_amount) as total_spent,
  ROUND(AVG(o.total_amount), 2) as avg_order_value
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.city, c.membership_tier
ORDER BY total_spent DESC;

In [0]:
%sql
-- Query: Product sales performance with category analysis
SELECT 
  p.category,
  p.product_name,
  COUNT(o.order_id) as times_ordered,
  SUM(o.quantity) as total_units_sold,
  SUM(o.total_amount) as total_revenue,
  ROUND(AVG(o.total_amount), 2) as avg_order_value
FROM products p
JOIN orders o ON p.product_id = o.product_id
GROUP BY p.category, p.product_name
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Query: Detailed order information for Premium customers
SELECT 
  o.order_id,
  o.order_date,
  c.name as customer_name,
  c.city,
  p.product_name,
  p.category,
  o.quantity,
  o.total_amount,
  ROUND(o.total_amount / o.quantity, 2) as unit_price
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE c.membership_tier = 'Premium'
ORDER BY o.order_date DESC, o.total_amount DESC;

In [0]:
%sql
-- Query: Find products that haven't been ordered yet
SELECT 
  p.product_id,
  p.product_name,
  p.category,
  p.price,
  p.stock_quantity
FROM products p
LEFT JOIN orders o ON p.product_id = o.product_id
WHERE o.order_id IS NULL
ORDER BY p.category, p.product_name;

In [0]:
%sql
-- Query: Monthly revenue and order trends
SELECT 
  DATE_TRUNC('MONTH', order_date) as month,
  COUNT(order_id) as total_orders,
  SUM(total_amount) as monthly_revenue,
  ROUND(AVG(total_amount), 2) as avg_order_value,
  COUNT(DISTINCT customer_id) as unique_customers
FROM orders
GROUP BY DATE_TRUNC('MONTH', order_date)
ORDER BY month;

In [0]:
%sql
-- Query: Rank customers by total spending using window functions
SELECT 
  c.customer_id,
  c.name,
  c.membership_tier,
  c.city,
  COALESCE(SUM(o.total_amount), 0) as total_spent,
  RANK() OVER (ORDER BY COALESCE(SUM(o.total_amount), 0) DESC) as spending_rank,
  DENSE_RANK() OVER (PARTITION BY c.membership_tier ORDER BY COALESCE(SUM(o.total_amount), 0) DESC) as tier_rank
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.membership_tier, c.city
ORDER BY spending_rank;

In [0]:
%sql
-- Query: Calculate total inventory value by category
SELECT 
  category,
  COUNT(*) as product_count,
  SUM(stock_quantity) as total_units,
  ROUND(SUM(price * stock_quantity), 2) as inventory_value,
  ROUND(AVG(price), 2) as avg_price_per_product
FROM products
GROUP BY category
ORDER BY inventory_value DESC;

## Working with Tables in Python and Scala

The following cells demonstrate how to access and query the sample tables using PySpark (Python) and Spark (Scala).

In [0]:
# Read the temp views into PySpark DataFrames
customers_df = spark.table("customers")
products_df = spark.table("products")
orders_df = spark.table("orders")

# Display schema
print("Customers Schema:")
customers_df.printSchema()

# Show sample data
print("\nFirst 3 customers:")
display(customers_df.limit(3))

In [0]:
from pyspark.sql import functions as F

# Filter customers from New York
ny_customers = customers_df.filter(F.col("city") == "New York")
print("Customers in New York:")
display(ny_customers)

# Aggregate: Total revenue by product category
revenue_by_category = (
    orders_df
    .join(products_df, "product_id")
    .groupBy("category")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("order_id").alias("order_count")
    )
    .orderBy(F.desc("total_revenue"))
)

print("\nRevenue by Category:")
display(revenue_by_category)

In [0]:
# Join all three tables to get complete order details
full_order_details = (
    orders_df
    .join(customers_df, "customer_id")
    .join(products_df, "product_id")
    .select(
        "order_id",
        "order_date",
        F.col("name").alias("customer_name"),
        "membership_tier",
        "product_name",
        "category",
        "quantity",
        "total_amount"
    )
    .orderBy(F.desc("order_date"))
)

print("Complete Order Details:")
display(full_order_details.limit(10))

# Calculate top spending customer using window functions
from pyspark.sql.window import Window

customer_spending = (
    orders_df
    .groupBy("customer_id")
    .agg(F.sum("total_amount").alias("total_spent"))
    .join(customers_df, "customer_id")
    .select("name", "city", "membership_tier", "total_spent")
    .orderBy(F.desc("total_spent"))
)

print("\nTop Customers by Spending:")
display(customer_spending)

## Creating Permanent Delta Tables in Unity Catalog

The following cells create permanent Delta tables from the temporary views, making them visible in the Unity Catalog.

In [0]:
%sql
-- We'll use the workspace.default schema for our permanent tables
-- This is the default location for user-created tables
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
-- Create permanent customers table
CREATE OR REPLACE TABLE workspace.default.customers
COMMENT 'Customer information including membership tiers'
AS
SELECT * FROM customers;

-- Show table info
DESCRIBE TABLE EXTENDED workspace.default.customers;

In [0]:
%sql
-- Create permanent products table
CREATE OR REPLACE TABLE workspace.default.products
COMMENT 'Product catalog with pricing and inventory'
AS
SELECT * FROM products;

-- Show table info
DESCRIBE TABLE EXTENDED workspace.default.products;

In [0]:
%sql
-- Create permanent orders table
CREATE OR REPLACE TABLE workspace.default.orders
COMMENT 'Order transactions linking customers and products'
AS
SELECT * FROM orders;

-- Show table info
DESCRIBE TABLE EXTENDED workspace.default.orders;

In [0]:
%sql
-- Verify all tables are created and visible in the catalog
SHOW TABLES IN workspace.default;

-- Query the permanent tables to confirm they work
SELECT 'Customers' as table_name, COUNT(*) as row_count FROM workspace.default.customers
UNION ALL
SELECT 'Products', COUNT(*) FROM workspace.default.products
UNION ALL
SELECT 'Orders', COUNT(*) FROM workspace.default.orders;

## ✅ Tables Successfully Created!

Your permanent Delta tables are now visible in Unity Catalog:
* **Location**: `workspace.default.customers`, `workspace.default.products`, `workspace.default.orders`
* **Format**: Delta Lake (optimized for analytics)
* **Type**: MANAGED (Databricks manages the data and metadata)

### How to Find Them:
1. **Catalog Explorer**: Click the Data icon in the left sidebar → Navigate to workspace → default → You'll see your three tables
2. **SQL Queries**: Reference them as `workspace.default.customers` (or just `customers` if workspace.default is your current schema)
3. **Python/Scala**: Use `spark.table("workspace.default.customers")`

These tables persist across sessions and are available to all users with appropriate permissions!

## Working with Volumes in Unity Catalog

**Volumes** provide a way to store and access non-tabular data (files) in Unity Catalog. They support any file type and integrate with Unity Catalog's governance features.

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# First, create a volume in Unity Catalog if it doesn't exist
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.default.sample_data
    COMMENT 'Volume for sample data files'
""")

print("Volume created: workspace.default.sample_data")

# Create sample sales data
sales_data = {
    'transaction_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    'date': ['2024-04-01', '2024-04-01', '2024-04-02', '2024-04-02', '2024-04-03',
             '2024-04-03', '2024-04-04', '2024-04-04', '2024-04-05', '2024-04-05'],
    'store_id': ['S001', 'S002', 'S001', 'S003', 'S002', 'S001', 'S003', 'S002', 'S001', 'S003'],
    'product_category': ['Electronics', 'Clothing', 'Electronics', 'Groceries', 'Clothing',
                         'Groceries', 'Electronics', 'Groceries', 'Clothing', 'Electronics'],
    'quantity': [2, 5, 1, 10, 3, 8, 1, 15, 4, 2],
    'unit_price': [299.99, 49.99, 1199.99, 5.99, 79.99, 12.99, 899.99, 3.99, 89.99, 449.99],
    'total_amount': [599.98, 249.95, 1199.99, 59.90, 239.97, 103.92, 899.99, 59.85, 359.96, 899.98],
    'payment_method': ['Credit Card', 'Cash', 'Credit Card', 'Debit Card', 'Credit Card',
                      'Cash', 'Credit Card', 'Debit Card', 'Credit Card', 'Cash']
}

# Create pandas DataFrame
df_pandas = pd.DataFrame(sales_data)

# Define the volume path
volume_path = '/Volumes/workspace/default/sample_data/sales_transactions.csv'

# Write CSV file to the volume
df_pandas.to_csv(volume_path, index=False)

print(f"\n✅ CSV file created successfully at: {volume_path}")
print(f"\nFile contains {len(df_pandas)} rows and {len(df_pandas.columns)} columns")
print("\nFirst 5 rows:")
print(df_pandas.head())

In [0]:
# Read the CSV file from the volume using Spark
df_spark = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/sample_data/sales_transactions.csv")

print("Reading CSV from Volume using Spark:")
print(f"Schema:")
df_spark.printSchema()

print("\nData preview:")
display(df_spark)

# Show some aggregations
print("\n📊 Sales Summary:")
summary = df_spark.groupBy("product_category") \
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_units_sold"),
        F.count("*").alias("transaction_count")
    ) \
    .orderBy(F.desc("total_revenue"))

display(summary)

In [0]:
# List all files in the volume
print("Files in volume workspace.default.sample_data:\n")
files = dbutils.fs.ls("/Volumes/workspace/default/sample_data/")

for file in files:
    size_mb = file.size / (1024 * 1024)
    print(f"  📄 {file.name}")
    print(f"     Path: {file.path}")
    print(f"     Size: {size_mb:.4f} MB")
    print()

# You can also read directly with pandas
import pandas as pd
df_check = pd.read_csv("/Volumes/workspace/default/sample_data/sales_transactions.csv")
print(f"\n✅ Verification: CSV contains {len(df_check)} rows")

print("\n" + "="*60)
print("ℹ️  Volume Information")
print("="*60)
print(f"Volume Path: /Volumes/workspace/default/sample_data/")
print(f"Full CSV Path: /Volumes/workspace/default/sample_data/sales_transactions.csv")
print(f"Access: Available from any notebook or job in your workspace")
print(f"Governance: Managed by Unity Catalog with access controls")

## 📋 Volume Creation Process - Documentation

### What We Accomplished

We successfully created a Unity Catalog volume and stored a sample CSV file in it. Here's the complete process that was followed:

---

### Step 1: Create the Volume in Unity Catalog

**What is a Volume?**
* A Unity Catalog object for storing unstructured data (files)
* Provides governed file storage with access controls
* Supports any file type: CSV, JSON, Parquet, images, PDFs, etc.
* Integrated with Unity Catalog's permissions and lineage tracking

**SQL Command Used:**
```sql
CREATE VOLUME IF NOT EXISTS workspace.default.sample_data
COMMENT 'Volume for sample data files'
```

**Key Points:**
* **Three-level namespace**: `catalog.schema.volume_name`
* **Location**: `workspace.default.sample_data`
* **IF NOT EXISTS**: Prevents errors if volume already exists
* **COMMENT**: Provides documentation visible in Catalog Explorer

---

### Step 2: Generate Sample Data

**Created a pandas DataFrame** with realistic sales transaction data:
* 10 transactions spanning 5 days (April 1-5, 2024)
* Fields: transaction_id, date, store_id, product_category, quantity, unit_price, total_amount, payment_method
* Multiple stores (S001, S002, S003)
* Multiple categories (Electronics, Clothing, Groceries)
* Various payment methods (Credit Card, Cash, Debit Card)

**Why pandas first?**
* Simple CSV writing with `to_csv()` method
* Easy to construct structured sample data
* Works seamlessly with volume paths

---

### Step 3: Write CSV File to Volume

**File Path Convention:**
```python
volume_path = '/Volumes/workspace/default/sample_data/sales_transactions.csv'
```

**Path Structure:**
* **Prefix**: `/Volumes/` (required for all volume paths)
* **Catalog**: `workspace`
* **Schema**: `default`
* **Volume Name**: `sample_data`
* **File Path**: `sales_transactions.csv` (can include subdirectories)

**Write Command:**
```python
df_pandas.to_csv(volume_path, index=False)
```

**Parameters:**
* `index=False`: Excludes pandas row index from CSV
* Direct path writing: No need for separate upload or copy operations

---

### Step 4: Read and Verify the CSV File

**Method 1: Spark DataFrame (Recommended for large files)**
```python
df_spark = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/sample_data/sales_transactions.csv")
```

**Benefits:**
* Distributed processing for large files
* Schema inference and validation
* Integration with Spark SQL and transformations
* Can write results to Delta tables

**Method 2: Pandas (Good for small files)**
```python
df_check = pd.read_csv("/Volumes/workspace/default/sample_data/sales_transactions.csv")
```

**Benefits:**
* Familiar pandas API
* Fast for small to medium files
* Easy data manipulation

---

### Step 5: List and Manage Files

**Using dbutils:**
```python
files = dbutils.fs.ls("/Volumes/workspace/default/sample_data/")
```

**File Information Retrieved:**
* File name
* Full path
* Size (in bytes, converted to MB)
* Modification time

**Other dbutils Operations:**
* `dbutils.fs.cp()`: Copy files
* `dbutils.fs.mv()`: Move/rename files
* `dbutils.fs.rm()`: Delete files
* `dbutils.fs.head()`: Preview file content

---

### Key Differences: Volumes vs Tables

| Aspect | Unity Catalog Tables | Unity Catalog Volumes |
|--------|---------------------|----------------------|
| **Data Type** | Structured (rows/columns) | Unstructured (files) |
| **Format** | Delta Lake, Parquet | Any file type |
| **Query** | SQL SELECT queries | File read operations |
| **Storage** | Optimized columnar | Raw file storage |
| **Use Cases** | Analytics, BI, ML features | Documents, images, raw data |
| **Optimization** | OPTIMIZE, Z-ORDER, clustering | Not applicable |
| **Schema** | Enforced schema | No schema enforcement |

---

### Best Practices for Volumes

#### 1. **Organize with a Clear Structure**
```
/Volumes/workspace/default/sample_data/
  ├── raw/
  │   ├── sales_transactions.csv
  │   └── customer_data.csv
  ├── processed/
  │   └── cleaned_sales.parquet
  └── archive/
      └── old_data.csv
```

#### 2. **Use Descriptive Volume Names**
* ✅ Good: `marketing_assets`, `ml_models`, `customer_documents`
* ❌ Avoid: `data`, `files`, `stuff`

#### 3. **Add Comments for Documentation**
```sql
CREATE VOLUME workspace.default.ml_models
COMMENT 'Trained ML models in MLflow format';
```

#### 4. **Set Appropriate Permissions**
```sql
-- Grant read access to data analysts
GRANT READ VOLUME ON workspace.default.sample_data TO `data_analysts`;

-- Grant write access to data engineers
GRANT WRITE VOLUME ON workspace.default.sample_data TO `data_engineers`;
```

#### 5. **Prefer Delta Tables for Analytical Data**
* Use volumes for: images, PDFs, raw logs, model artifacts
* Use tables for: structured data that will be queried with SQL

#### 6. **Consider File Formats**
* **CSV**: Human-readable, widely compatible, larger size
* **Parquet**: Compressed, columnar, fast for analytics
* **JSON**: Flexible schema, good for semi-structured data
* **Avro**: Schema evolution support, good for streaming

---

### Common Patterns

#### Pattern 1: Landing Zone for Raw Files
```python
# Files arrive in volume, then get processed into Delta tables
df = spark.read.csv("/Volumes/workspace/default/landing_zone/new_data.csv")
df.write.mode("append").saveAsTable("workspace.default.raw_data")
```

#### Pattern 2: ML Model Storage
```python
import mlflow

# Log model to volume
mlflow.sklearn.log_model(
    model,
    artifact_path="model",
    registered_model_name="my_model"
)

# Models are stored in volumes managed by MLflow
```

#### Pattern 3: Document Storage for AI
```python
# Store PDFs in volume for RAG applications
pdfs_path = "/Volumes/workspace/default/documents/contracts/"

# Process with AI functions
spark.sql(f"""
  SELECT ai_extract_text('{pdfs_path}/contract_001.pdf') as contract_text
""")
```

---

### Accessing Volumes

#### From Notebooks (Python)
```python
# Direct pandas read/write
import pandas as pd
df = pd.read_csv("/Volumes/catalog/schema/volume/file.csv")
df.to_parquet("/Volumes/catalog/schema/volume/file.parquet")

# Spark read/write
df = spark.read.parquet("/Volumes/catalog/schema/volume/data/")
df.write.format("delta").save("/Volumes/catalog/schema/volume/output/")
```

#### From SQL
```sql
-- Read CSV from volume into a table
CREATE TABLE workspace.default.sales_from_volume AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/sample_data/sales_transactions.csv',
  format => 'csv',
  header => true
);
```

#### From Jobs and Pipelines
* Volume paths work the same across all Databricks contexts
* No special configuration needed
* Same governance and permissions apply

---

### Monitoring and Governance

#### View Volume Details
```sql
DESCRIBE VOLUME workspace.default.sample_data;
SHOW VOLUMES IN workspace.default;
```

#### Check Permissions
```sql
SHOW GRANTS ON VOLUME workspace.default.sample_data;
```

#### Audit Access
* Volume access is logged in Unity Catalog audit logs
* Track who accessed which files and when
* Available through System Tables: `system.access.audit`

---

### Summary of Our Implementation

✅ **Created**: Volume `workspace.default.sample_data`  
✅ **Stored**: CSV file with 10 sales transactions  
✅ **Verified**: File readable via Spark and pandas  
✅ **Analyzed**: Aggregated sales by product category  
✅ **Listed**: Confirmed file exists with proper metadata  

**File Location**: `/Volumes/workspace/default/sample_data/sales_transactions.csv`  
**File Size**: ~0.0005 MB (approximately 500 bytes)  
**Contents**: 10 rows, 8 columns  
**Format**: CSV with headers  

---

### Next Steps You Can Try

1. **Add more files** to the volume
   ```python
   df.to_json("/Volumes/workspace/default/sample_data/sales.json")
   df.to_parquet("/Volumes/workspace/default/sample_data/sales.parquet")
   ```

2. **Create a Delta table** from the CSV
   ```sql
   CREATE TABLE workspace.default.sales_from_volume AS
   SELECT * FROM read_files('/Volumes/workspace/default/sample_data/sales_transactions.csv',
                            format => 'csv', header => true);
   ```

3. **Set up Auto Loader** to process files as they arrive
   ```python
   df = spark.readStream.format("cloudFiles") \
       .option("cloudFiles.format", "csv") \
       .load("/Volumes/workspace/default/sample_data/")
   ```

4. **Organize with subdirectories**
   ```python
   dbutils.fs.mkdirs("/Volumes/workspace/default/sample_data/archive/")
   dbutils.fs.cp("source", "/Volumes/workspace/default/sample_data/archive/")
   ```

# Lakeflow Connect: Managed Data Ingestion in Databricks

## Overview

**Lakeflow Connect** is Databricks' fully managed data ingestion service that simplifies moving data from external sources into Delta Lake tables in Unity Catalog. It provides a no-code/low-code interface to set up reliable, scalable data pipelines without writing ETL code.

## Key Features

### 1. **Fully Managed Service**
* No infrastructure management required
* Automatic scaling based on data volume
* Built-in error handling and retry logic
* Monitoring and observability out of the box

### 2. **Native Unity Catalog Integration**
* Data lands directly as Delta tables in Unity Catalog
* Automatic schema evolution and inference
* Built-in data governance and lineage tracking
* Centralized access control and auditing

### 3. **Change Data Capture (CDC)**
* Automatically captures inserts, updates, and deletes from source systems
* Maintains slowly changing dimensions (SCD Type 1 and Type 2)
* Preserves data history for compliance and analytics
* Efficient incremental processing

### 4. **Enterprise Connectors**
Supports a wide range of data sources including:
* **Databases**: MySQL, PostgreSQL, SQL Server, Oracle, DB2
* **SaaS Applications**: Salesforce, Google Analytics, Adobe Analytics, HubSpot, Workday
* **Cloud Storage**: Amazon S3, Azure Blob Storage, Google Cloud Storage
* **Event Streams**: Kafka, Amazon Kinesis, Azure Event Hubs
* **Data Warehouses**: Snowflake, Amazon Redshift, Google BigQuery

## Architecture

```
┌─────────────────┐
│  Source System  │ (Database, SaaS, Cloud Storage)
└────────┬────────┘
         │
         │ Lakeflow Connect
         │ (Managed Ingestion)
         │
         ▼
┌─────────────────┐
│  Delta Tables   │ (Unity Catalog)
│  Bronze Layer   │
└────────┬────────┘
         │
         │ Lakeflow Pipelines
         │ (Transformation)
         │
         ▼
┌─────────────────┐
│ Silver & Gold   │ (Refined Data)
│    Layers       │
└─────────────────┘
```

## Common Use Cases

### 1. **Database Replication**
Replicate operational databases to Databricks for analytics without impacting production workloads.
* Full table snapshots
* Incremental CDC-based sync
* Schema change detection

### 2. **SaaS Data Integration**
Consolidate data from multiple SaaS applications into a unified analytics platform.
* Marketing data (Google Analytics, Adobe, HubSpot)
* Sales data (Salesforce, Dynamics 365)
* Financial data (Workday, NetSuite)

### 3. **Cloud Storage Ingestion**
Automatically ingest files from cloud storage buckets as they arrive.
* JSON, CSV, Parquet, Avro files
* Auto-detection of schema
* Directory-based partitioning

### 4. **Real-Time Streaming**
Ingest streaming data from event platforms for real-time analytics.
* Kafka topics
* Kinesis streams
* Event Hubs

## How It Works

### Step 1: Configure Connection
* Select your data source type
* Provide connection credentials (stored securely in Secrets)
* Test connectivity

### Step 2: Select Data
* Choose tables, datasets, or file patterns to ingest
* Configure filters and transformations (optional)
* Set up CDC mode (snapshot vs incremental)

### Step 3: Configure Destination
* Specify target catalog and schema in Unity Catalog
* Choose table naming conventions
* Set partitioning and optimization settings

### Step 4: Schedule & Monitor
* Set ingestion frequency (continuous, hourly, daily, etc.)
* Configure alerts for failures or SLA breaches
* Monitor progress through built-in dashboard

## Benefits

### For Data Engineers
✅ **Reduced Development Time**: No need to write custom extraction code  
✅ **Consistent Patterns**: Standardized approach across all data sources  
✅ **Less Maintenance**: Automatic handling of schema changes and connector updates  
✅ **Better Reliability**: Built-in retry logic and error handling  

### For Data Teams
✅ **Faster Time-to-Insight**: Data available in minutes, not days  
✅ **Unified Data Platform**: All data in one place (Unity Catalog)  
✅ **Better Data Quality**: Automatic validation and schema enforcement  
✅ **Clear Lineage**: Track data from source to analytics  

### For Organizations
✅ **Lower TCO**: Reduce infrastructure and maintenance costs  
✅ **Enhanced Security**: Centralized governance with Unity Catalog  
✅ **Compliance Ready**: Audit logs and access controls built-in  
✅ **Scalability**: Handles growing data volumes automatically  

## Best Practices

### 1. **Organize with Catalogs and Schemas**
```sql
-- Create dedicated schemas for raw ingested data
CREATE SCHEMA IF NOT EXISTS main.bronze_salesforce;
CREATE SCHEMA IF NOT EXISTS main.bronze_mysql;
```

### 2. **Use CDC for Large Tables**
* Enables incremental processing
* Reduces ingestion time and costs
* Preserves historical changes

### 3. **Set Up Monitoring and Alerts**
* Configure notifications for failed ingestions
* Monitor data freshness SLAs
* Track ingestion metrics (rows processed, latency)

### 4. **Implement Data Quality Checks**
```python
# Example: Validate ingested data in downstream pipeline
from pyspark.sql import functions as F

# Check for nulls in critical columns
df = spark.table("main.bronze_salesforce.accounts")
quality_check = df.filter(F.col("account_id").isNull()).count()
if quality_check > 0:
    raise ValueError(f"Found {quality_check} rows with null account_id")
```

### 5. **Leverage Medallion Architecture**
* **Bronze** (Raw): Use Lakeflow Connect to land data as-is
* **Silver** (Cleansed): Use Lakeflow Spark Declarative Pipelines for transformations
* **Gold** (Aggregated): Build business-level aggregates for BI tools

## Accessing Lakeflow Connect

### Via Databricks UI
1. Navigate to **Workflows** → **Lakeflow Ingestion**
2. Click **Create ingestion configuration**
3. Follow the wizard to configure source, destination, and schedule

### Via API (for automation)
```python
import requests

# Example: Create an ingestion configuration via REST API
url = f"{databricks_instance}/api/2.0/pipelines"
headers = {"Authorization": f"Bearer {access_token}"}
payload = {
    "name": "Salesforce to Unity Catalog",
    "source": {
        "connector_type": "salesforce",
        "credentials_id": "salesforce_creds",
        "objects": ["Account", "Opportunity", "Contact"]
    },
    "destination": {
        "catalog": "main",
        "schema": "bronze_salesforce"
    },
    "schedule": {"quartz_cron_expression": "0 0 * * * ?"}
}

response = requests.post(url, headers=headers, json=payload)
```

## When to Use Lakeflow Connect

### ✅ Best For:
* Replicating entire databases or SaaS data stores
* Continuous CDC-based synchronization
* Standard connector-based ingestion (no custom logic)
* Quick proof-of-concepts and MVPs

### ⚠️ Consider Alternatives For:
* Highly custom extraction logic → Use notebooks with **Auto Loader**
* Complex pre-ingestion transformations → Use **Lakeflow Spark Declarative Pipelines**
* One-time bulk imports → Use **COPY INTO** or **dbutils.fs**
* Sources without pre-built connectors → Use Python/Scala with REST APIs

## Resources

* **Documentation**: [Databricks Lakeflow Connect Docs](https://docs.databricks.com)
* **Connector List**: Check available connectors in the Databricks UI
* **Pricing**: Based on data volume processed (see Databricks pricing page)
* **Support**: Available through Databricks support channels

---

**Related Technologies:**
* **Auto Loader**: For custom file-based ingestion with schema evolution
* **Lakeflow Spark Declarative Pipelines (SDP)**: For data transformation and quality
* **Delta Live Tables**: For building reliable data pipelines
* **Unity Catalog**: For data governance and access control

# Lakeflow Spark Declarative Pipelines (SDP)

## Overview

**Lakeflow Spark Declarative Pipelines** (formerly Delta Live Tables or DLT) is a declarative framework for building reliable, maintainable, and testable data pipelines on Databricks. Instead of writing complex orchestration code, you declare what you want (the desired state) and the platform handles how to achieve it.

### Key Concept: Declarative vs Imperative

**Imperative** (Traditional ETL):
```python
# You specify HOW to do everything
df = spark.read.parquet("s3://bucket/data")
df_filtered = df.filter(col("status") == "active")
df_filtered.write.mode("append").saveAsTable("my_table")
# You handle: incremental logic, deduplication, retries, dependencies
```

**Declarative** (Spark Declarative Pipelines):
```python
# You specify WHAT you want
@dlt.table(
    comment="Active users from source data",
    table_properties={"quality": "gold"}
)
def active_users():
    return spark.read.parquet("s3://bucket/data").filter(col("status") == "active")
# Platform handles: incremental processing, dependencies, retries, optimization
```

## Core Concepts

### 1. **Tables**
The fundamental building blocks representing datasets at different stages.

#### Streaming Tables
```python
import dlt
from pyspark.sql import functions as F

@dlt.table(
    comment="Raw clickstream events",
    table_properties={"quality": "bronze"}
)
def clickstream_raw():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/mnt/raw/clickstream/")
    )
```

#### Materialized Views
```python
@dlt.table(
    comment="Aggregated daily user activity"
)
def daily_user_activity():
    return (
        dlt.read("clickstream_raw")
        .groupBy("user_id", F.to_date("timestamp").alias("date"))
        .agg(
            F.count("*").alias("event_count"),
            F.countDistinct("page_id").alias("pages_visited")
        )
    )
```

### 2. **Data Quality Expectations**
Define data quality rules that are enforced automatically.

```python
@dlt.table
@dlt.expect("valid_email", "email IS NOT NULL AND email LIKE '%@%.%'")
@dlt.expect_or_drop("valid_timestamp", "timestamp > '2020-01-01'")
@dlt.expect_or_fail("required_fields", "user_id IS NOT NULL AND event_type IS NOT NULL")
def validated_events():
    return dlt.read("clickstream_raw")
```

**Expectation Actions:**
* `@dlt.expect` - Log violations, allow records through
* `@dlt.expect_or_drop` - Drop records that violate
* `@dlt.expect_or_fail` - Fail pipeline if any violations

### 3. **Change Data Capture (CDC)**
Automatically apply changes from CDC feeds.

```python
# Apply CDC changes to target table
dlt.create_streaming_table("customers")

dlt.apply_changes(
    target="customers",
    source="customers_cdc_raw",
    keys=["customer_id"],
    sequence_by="update_timestamp",
    stored_as_scd_type="2",  # or "1" for overwrite
    except_column_list=["_rescued_data"]
)
```

### 4. **Auto Loader Integration**
Efficiently ingest files as they arrive in cloud storage.

```python
@dlt.table
def raw_events():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/mnt/schema/events")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/mnt/landing/events/")
    )
```

## Pipeline Architecture

### Medallion Architecture Pattern

```
┌──────────────────────────────────────────────┐
│  Bronze Layer (Raw Data)                     │
│  - Exact copy of source data                 │
│  - Minimal transformations                   │
│  - Streaming tables for real-time ingestion  │
└────────────────┬─────────────────────────────┘
                 │
                 ▼
┌──────────────────────────────────────────────┐
│  Silver Layer (Cleansed & Conformed)         │
│  - Data quality checks applied               │
│  - Standardized formats                      │
│  - Deduplicated and validated                │
└────────────────┬─────────────────────────────┘
                 │
                 ▼
┌──────────────────────────────────────────────┐
│  Gold Layer (Business-Level Aggregates)      │
│  - Aggregated metrics                        │
│  - Feature tables for ML                     │
│  - Ready for BI and analytics                │
└──────────────────────────────────────────────┘
```

### Example: Complete Medallion Pipeline

```python
import dlt
from pyspark.sql import functions as F

# BRONZE: Raw data ingestion
@dlt.table(
    comment="Raw orders from source system",
    table_properties={"quality": "bronze"}
)
def orders_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/mnt/raw/orders/")
    )

# SILVER: Cleansed and validated
@dlt.table(
    comment="Validated orders with quality checks",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_drop("positive_amount", "amount > 0")
@dlt.expect("recent_order", "order_date >= current_date() - INTERVAL 2 YEARS")
def orders_silver():
    return (
        dlt.read_stream("orders_bronze")
        .select(
            "order_id",
            "customer_id",
            F.to_timestamp("order_date").alias("order_date"),
            F.col("amount").cast("decimal(10,2)").alias("amount"),
            "status"
        )
        .dropDuplicates(["order_id"])
    )

# GOLD: Business aggregates
@dlt.table(
    comment="Daily revenue by customer segment",
    table_properties={"quality": "gold"}
)
def daily_revenue_by_segment():
    orders = dlt.read("orders_silver")
    customers = dlt.read("customers_silver")
    
    return (
        orders
        .join(customers, "customer_id")
        .groupBy(
            F.to_date("order_date").alias("date"),
            "customer_segment"
        )
        .agg(
            F.sum("amount").alias("total_revenue"),
            F.count("order_id").alias("order_count"),
            F.countDistinct("customer_id").alias("unique_customers")
        )
    )
```

## Key Features

### 1. **Automatic Dependency Management**
* Pipelines automatically determine execution order
* Tables are computed only when dependencies are ready
* Handles complex DAGs without manual orchestration

### 2. **Incremental Processing**
```python
# Automatically processes only new data
@dlt.table
def incremental_aggregates():
    return (
        dlt.read_stream("source_table")
        .groupBy("date", "category")
        .agg(F.sum("amount").alias("total"))
    )
```

### 3. **Schema Evolution**
* Automatically handles new columns in source data
* Validates schema changes against expectations
* Preserves backward compatibility

### 4. **Built-in Monitoring**
* Real-time pipeline execution metrics
* Data quality metrics tracked automatically
* Event logs for debugging and auditing
* Lineage tracking from source to gold

### 5. **Testing and Development**
```python
# Development mode: Fast iteration with sample data
# Production mode: Full data processing with optimizations

# Use different configurations for dev vs prod
if spark.conf.get("pipeline.mode") == "development":
    sample_fraction = 0.01
else:
    sample_fraction = 1.0
```

## Configuration

### Pipeline Settings
```json
{
  "name": "customer_analytics_pipeline",
  "storage": "/mnt/pipelines/customer_analytics",
  "target": "main.analytics",
  "libraries": [
    {"notebook": {"path": "/Pipelines/bronze_layer"}},
    {"notebook": {"path": "/Pipelines/silver_layer"}},
    {"notebook": {"path": "/Pipelines/gold_layer"}}
  ],
  "clusters": [
    {
      "label": "default",
      "num_workers": 2,
      "node_type_id": "i3.xlarge"
    }
  ],
  "continuous": true,
  "channel": "CURRENT"
}
```

## Best Practices

### 1. **Organize by Layer**
```
/Pipelines/
  ├── bronze/
  │   ├── ingest_orders.py
  │   ├── ingest_customers.py
  │   └── ingest_products.py
  ├── silver/
  │   ├── cleanse_orders.py
  │   └── cleanse_customers.py
  └── gold/
      ├── daily_metrics.py
      └── customer_segments.py
```

### 2. **Use Expectations Wisely**
```python
# Critical fields: Fail pipeline
@dlt.expect_or_fail("primary_key", "id IS NOT NULL")

# Data quality: Drop bad records
@dlt.expect_or_drop("valid_email", "email RLIKE '^[^@]+@[^@]+\\.[^@]+$'")

# Monitoring: Log but allow
@dlt.expect("recent_data", "date >= current_date() - INTERVAL 7 DAYS")
```

### 3. **Leverage Streaming for Real-Time**
```python
# Streaming for continuous processing
@dlt.table
def real_time_alerts():
    return (
        dlt.read_stream("events_bronze")
        .filter(F.col("event_type") == "critical_error")
        .select("timestamp", "user_id", "error_message")
    )
```

### 4. **Add Metadata and Comments**
```python
@dlt.table(
    comment="Customer lifetime value calculation",
    table_properties={
        "quality": "gold",
        "owner": "analytics-team",
        "pii": "false"
    }
)
```

## Development Workflow

### 1. **Create Pipeline Notebook**
```python
import dlt
from pyspark.sql import functions as F

# Define your tables and transformations
@dlt.table
def my_table():
    return spark.read.parquet("/mnt/data/source")
```

### 2. **Create Pipeline in UI**
* Navigate to **Workflows** → **Lakeflow Pipelines**
* Click **Create Pipeline**
* Configure settings (name, target catalog, notebooks)
* Set compute resources and schedule

### 3. **Run in Development Mode**
* Fast iteration with sample data
* Validate logic and data quality rules
* Debug with detailed logs

### 4. **Deploy to Production**
* Switch to continuous or scheduled mode
* Enable auto-scaling for production workloads
* Set up alerts for failures

## Monitoring and Observability

### Pipeline Metrics
* **Data flow**: Records processed per second
* **Data quality**: Expectation pass/fail rates
* **Performance**: Processing latency and throughput
* **Errors**: Failed updates and their causes

### Accessing Metrics
```python
# Query pipeline event logs
events = spark.read.format("delta").load(
    "dbfs:/pipelines/<pipeline-id>/system/events"
)

# Analyze data quality
quality_metrics = events.filter(
    F.col("details").getField("flow_definition").isNotNull()
)

display(quality_metrics)
```

## When to Use Spark Declarative Pipelines

### ✅ Best For:
* Medallion architecture (Bronze/Silver/Gold)
* Continuous streaming pipelines
* Complex data quality requirements
* Production ETL/ELT workflows
* CDC-based synchronization
* Real-time data transformations

### ⚠️ Consider Alternatives For:
* Ad-hoc data exploration → Use notebooks
* One-time data migration → Use batch jobs
* Simple file copies → Use Lakeflow Connect
* ML model training → Use MLflow and Jobs

## Common Patterns

### Pattern 1: Slowly Changing Dimensions (SCD Type 2)
```python
dlt.create_streaming_table("customers")

dlt.apply_changes(
    target="customers",
    source="customer_changes",
    keys=["customer_id"],
    sequence_by="update_timestamp",
    stored_as_scd_type="2"
)
```

### Pattern 2: Real-Time Aggregations
```python
@dlt.table
def real_time_metrics():
    return (
        dlt.read_stream("events")
        .withWatermark("timestamp", "10 minutes")
        .groupBy(
            F.window("timestamp", "5 minutes"),
            "event_type"
        )
        .agg(F.count("*").alias("event_count"))
    )
```

### Pattern 3: Data Deduplication
```python
@dlt.table
def deduplicated_orders():
    return (
        dlt.read_stream("orders_raw")
        .withWatermark("order_timestamp", "1 hour")
        .dropDuplicates(["order_id"])
    )
```

## Resources

* **Documentation**: [Databricks Spark Declarative Pipelines](https://docs.databricks.com/delta-live-tables)
* **Best Practices**: [Pipeline Design Patterns](https://docs.databricks.com/delta-live-tables/patterns.html)
* **API Reference**: [Python DLT API](https://docs.databricks.com/delta-live-tables/python-ref.html)

---

**Related Technologies:**
* **Lakeflow Connect**: For managed data ingestion
* **Delta Lake**: Underlying storage format
* **Unity Catalog**: For governance and lineage
* **Lakeflow Jobs**: For pipeline orchestration and scheduling

# Lakeflow Jobs: Workflow Orchestration in Databricks

## Overview

**Lakeflow Jobs** (formerly Databricks Workflows) is the orchestration layer for scheduling and managing data pipelines, machine learning workflows, and ETL processes in Databricks. It provides a reliable, scalable way to run tasks on a schedule or trigger them based on events.

## Key Features

### 1. **Multi-Task Workflows**
Orchestrate complex workflows with multiple dependent tasks.

```
┌──────────────────────────────────────────────┐
│  Task 1: Ingest Data (Notebook)             │
└──────────────┬────────────────────────────────┘
               │
               ▼
┌──────────────────────────────────────────────┐
│  Task 2: Transform (Spark Declarative)      │
└──────────────┬────────────────────────────────┘
               │
       ┌───────┼───────┐
       │               │
       ▼               ▼
┌───────────┐   ┌──────────────────┐
│ Task 3a:  │   │  Task 3b: ML     │
│ Load to   │   │  Training        │
│ Data      │   │  (Python)        │
│ Warehouse │   └──────────────────┘
└───────────┘
```

### 2. **Multiple Task Types**
Support for diverse workload types in a single workflow.

* **Notebooks** - Python, SQL, Scala, R notebooks
* **Python Files** - Standalone Python scripts
* **JAR Tasks** - Scala/Java applications
* **Spark Submit** - Apache Spark applications
* **Spark Declarative Pipelines** - Run data pipelines
* **dbt Projects** - Execute dbt transformations
* **SQL Queries** - Run SQL warehouses queries
* **Python Wheels** - Custom Python packages

### 3. **Flexible Scheduling**
* **Cron-based** - Traditional time-based scheduling
* **File arrival** - Trigger when new files appear
* **External systems** - API-triggered execution
* **Continuous** - Run as soon as previous execution completes
* **On-demand** - Manual triggers from UI or API

### 4. **Compute Management**
* **Job clusters** - Ephemeral clusters created per run (cost-efficient)
* **All-purpose clusters** - Shared interactive clusters
* **Serverless** - No cluster management required
* **Pools** - Pre-warmed instances for faster starts

## Creating a Job

### Via UI

1. Navigate to **Workflows** in the sidebar
2. Click **Create Job**
3. Configure:
   - **Job name**: Descriptive name for the workflow
   - **Tasks**: Add one or more tasks
   - **Schedule**: Define when to run
   - **Cluster**: Select compute resources
   - **Parameters**: Pass runtime parameters
   - **Alerts**: Configure notifications

### Via API

```python
import requests

url = f"{databricks_instance}/api/2.1/jobs/create"
headers = {"Authorization": f"Bearer {token}"}

payload = {
    "name": "Daily ETL Pipeline",
    "tasks": [
        {
            "task_key": "ingest_data",
            "notebook_task": {
                "notebook_path": "/ETL/ingest",
                "base_parameters": {
                    "date": "{{job.start_time.date}}"
                }
            },
            "new_cluster": {
                "spark_version": "13.3.x-scala2.12",
                "node_type_id": "i3.xlarge",
                "num_workers": 2
            }
        },
        {
            "task_key": "transform_data",
            "depends_on": [{"task_key": "ingest_data"}],
            "pipeline_task": {
                "pipeline_id": "<pipeline-id>"
            }
        },
        {
            "task_key": "load_warehouse",
            "depends_on": [{"task_key": "transform_data"}],
            "sql_task": {
                "warehouse_id": "<warehouse-id>",
                "query": {
                    "query_id": "<query-id>"
                }
            }
        }
    ],
    "schedule": {
        "quartz_cron_expression": "0 0 2 * * ?",
        "timezone_id": "America/Los_Angeles"
    },
    "email_notifications": {
        "on_failure": ["data-team@company.com"]
    },
    "max_concurrent_runs": 1
}

response = requests.post(url, headers=headers, json=payload)
job_id = response.json()["job_id"]
print(f"Created job: {job_id}")
```

### Via Databricks Asset Bundles (DABs)

```yaml
# databricks.yml
resources:
  jobs:
    daily_etl_pipeline:
      name: Daily ETL Pipeline
      
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "America/Los_Angeles"
      
      tasks:
        - task_key: ingest_data
          notebook_task:
            notebook_path: ../notebooks/ingest.py
            base_parameters:
              env: "${bundle.target}"
          new_cluster:
            spark_version: 13.3.x-scala2.12
            node_type_id: i3.xlarge
            num_workers: 2
        
        - task_key: transform_data
          depends_on:
            - task_key: ingest_data
          pipeline_task:
            pipeline_id: ${resources.pipelines.bronze_to_gold.id}
        
        - task_key: load_warehouse
          depends_on:
            - task_key: transform_data
          sql_task:
            warehouse_id: ${var.warehouse_id}
            query:
              query_id: ${resources.queries.load_to_warehouse.id}
      
      email_notifications:
        on_failure:
          - data-team@company.com
```

Deploy with: `databricks bundle deploy --target prod`

## Task Configuration

### Notebook Task

```python
{
    "task_key": "data_processing",
    "notebook_task": {
        "notebook_path": "/Shared/ETL/process_data",
        "base_parameters": {
            "input_path": "/mnt/raw/data",
            "output_table": "analytics.processed_data",
            "run_date": "{{job.start_time.date}}"
        }
    },
    "libraries": [
        {"pypi": {"package": "pandas==2.0.0"}},
        {"pypi": {"package": "scikit-learn==1.3.0"}}
    ],
    "timeout_seconds": 3600,
    "max_retries": 2,
    "retry_on_timeout": True
}
```

### Python Task

```python
{
    "task_key": "ml_training",
    "python_task": {
        "python_file": "/Workspace/ML/train_model.py",
        "parameters": ["--model-type", "xgboost", "--epochs", "100"]
    },
    "new_cluster": {
        "spark_version": "13.3.x-ml-scala2.12",
        "node_type_id": "g4dn.xlarge",
        "num_workers": 4,
        "spark_conf": {
            "spark.databricks.cluster.profile": "singleNode"
        }
    }
}
```

### Pipeline Task

```python
{
    "task_key": "run_pipeline",
    "pipeline_task": {
        "pipeline_id": "abc123-pipeline-id",
        "full_refresh": False
    }
}
```

### SQL Task

```python
{
    "task_key": "refresh_dashboard",
    "sql_task": {
        "warehouse_id": "xyz789-warehouse-id",
        "query": {
            "query_id": "query-uuid-here"
        },
        "parameters": {
            "start_date": "{{job.start_time.date}}",
            "end_date": "{{job.end_time.date}}"
        }
    }
}
```

## Task Dependencies

### Linear Dependencies

```python
tasks = [
    {"task_key": "task_a"},
    {"task_key": "task_b", "depends_on": [{"task_key": "task_a"}]},
    {"task_key": "task_c", "depends_on": [{"task_key": "task_b"}]}
]
```

### Parallel Execution

```python
tasks = [
    {"task_key": "ingest"},
    {"task_key": "process_a", "depends_on": [{"task_key": "ingest"}]},
    {"task_key": "process_b", "depends_on": [{"task_key": "ingest"}]},
    {"task_key": "process_c", "depends_on": [{"task_key": "ingest"}]},
    {
        "task_key": "aggregate",
        "depends_on": [
            {"task_key": "process_a"},
            {"task_key": "process_b"},
            {"task_key": "process_c"}
        ]
    }
]
```

### Conditional Execution

```python
{
    "task_key": "validate_data",
    "depends_on": [{"task_key": "ingest"}]
}
{
    "task_key": "process_if_valid",
    "depends_on": [{
        "task_key": "validate_data",
        "outcome": "success"  # Only run if validation succeeds
    }]
}
{
    "task_key": "alert_if_invalid",
    "depends_on": [{
        "task_key": "validate_data",
        "outcome": "failed"  # Only run if validation fails
    }]
}
```

## Parameters and Variables

### Job Parameters

Pass parameters at runtime:

```python
# Define parameter in notebook
dbutils.widgets.text("run_date", "2024-01-01")
run_date = dbutils.widgets.get("run_date")

print(f"Processing data for {run_date}")
```

### Built-in Variables

```python
# Available in all tasks
{
    "job.start_time.date": "2024-04-23",
    "job.start_time.timestamp": "2024-04-23T10:30:00Z",
    "job.id": "123456",
    "job.run_id": "789",
    "job.trigger": "PERIODIC_SCHEDULE"
}

# Use in task parameters
"base_parameters": {
    "processing_date": "{{job.start_time.date}}"
}
```

### Task Values (Passing Data Between Tasks)

```python
# In first task (notebook or Python)
dbutils.jobs.taskValues.set(key="record_count", value=1000)
dbutils.jobs.taskValues.set(key="output_path", value="/mnt/output/data")

# In dependent task
record_count = dbutils.jobs.taskValues.get(
    taskKey="previous_task",
    key="record_count"
)

output_path = dbutils.jobs.taskValues.get(
    taskKey="previous_task",
    key="output_path"
)

print(f"Processing {record_count} records from {output_path}")
```

## Scheduling

### Cron Expressions (Quartz Format)

```
Format: second minute hour day month weekday year

Examples:
0 0 8 * * ? *          # Every day at 8 AM
0 30 10 ? * MON *      # Every Monday at 10:30 AM
0 0 */4 * * ? *        # Every 4 hours
0 0 0 1 * ? *          # First day of every month at midnight
0 0 2 ? * MON-FRI *    # Weekdays at 2 AM
```

### File Arrival Trigger

```python
{
    "file_arrival": {
        "url": "s3://my-bucket/landing/data/",
        "min_time_between_triggers_seconds": 300,
        "wait_after_last_change_seconds": 60
    }
}
```

## Error Handling

### Retry Configuration

```python
{
    "task_key": "flaky_task",
    "max_retries": 3,
    "min_retry_interval_millis": 60000,  # Wait 1 minute between retries
    "retry_on_timeout": True
}
```

### Timeout Settings

```python
{
    "timeout_seconds": 7200,  # Task-level timeout (2 hours)
    "job_timeout_seconds": 86400  # Job-level timeout (24 hours)
}
```

### Failure Handling

```python
tasks = [
    {"task_key": "critical_task"},
    {
        "task_key": "cleanup",
        "depends_on": [{
            "task_key": "critical_task",
            "outcome": "failed"
        }],
        # This runs ONLY if critical_task fails
    }
]
```

## Monitoring and Alerts

### Email Notifications

```python
{
    "email_notifications": {
        "on_start": ["team-lead@company.com"],
        "on_success": ["team@company.com"],
        "on_failure": ["team@company.com", "oncall@company.com"],
        "on_duration_warning_threshold_exceeded": ["team-lead@company.com"]
    },
    "notification_settings": {
        "no_alert_for_skipped_runs": True,
        "no_alert_for_canceled_runs": False
    }
}
```

### Webhook Notifications

```python
{
    "webhook_notifications": {
        "on_failure": [
            {
                "id": "slack-webhook-id"
            }
        ]
    }
}
```

### Health Monitoring

```python
{
    "health": {
        "rules": [
            {
                "metric": "RUN_DURATION_SECONDS",
                "op": "GREATER_THAN",
                "value": 3600
            }
        ]
    }
}
```

## Best Practices

### 1. **Use Job Clusters for Production**
* More cost-effective than all-purpose clusters
* Automatic cleanup after job completes
* Isolated compute for reliability

```python
{
    "new_cluster": {
        "spark_version": "13.3.x-scala2.12",
        "node_type_id": "i3.xlarge",
        "num_workers": 2,
        "autoscale": {
            "min_workers": 2,
            "max_workers": 8
        }
    }
}
```

### 2. **Implement Idempotency**

```python
# Make tasks rerunnable without side effects
def process_data(run_date):
    # Use CREATE OR REPLACE for tables
    spark.sql(f"""
        CREATE OR REPLACE TABLE analytics.daily_metrics
        PARTITION BY (date)
        AS SELECT * FROM source WHERE date = '{run_date}'
    """)
```

### 3. **Organize with Task Libraries**

```python
# Shared library configuration
common_libraries = [
    {"pypi": {"package": "pandas==2.0.0"}},
    {"pypi": {"package": "numpy==1.24.0"}}
]

tasks = [
    {
        "task_key": "task1",
        "libraries": common_libraries
    },
    {
        "task_key": "task2",
        "libraries": common_libraries
    }
]
```

### 4. **Use Parameters for Flexibility**

```python
# Parameterize dates, paths, and configurations
"base_parameters": {
    "env": "${var.environment}",
    "start_date": "{{job.start_time.date}}",
    "catalog": "${var.catalog_name}"
}
```

### 5. **Monitor Job Runs Programmatically**

```python
import requests

def check_job_status(job_id, run_id):
    url = f"{databricks_instance}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers, params=params)
    run_info = response.json()
    
    state = run_info["state"]["life_cycle_state"]
    result = run_info["state"].get("result_state")
    
    return state, result
```

### 6. **Leverage Asset Bundles for CI/CD**

```bash
# Development
databricks bundle deploy --target dev
databricks bundle run daily_etl_pipeline --target dev

# Production
databricks bundle deploy --target prod
databricks bundle run daily_etl_pipeline --target prod
```

## Common Patterns

### Pattern 1: Bronze-Silver-Gold ETL

```python
tasks = [
    {
        "task_key": "ingest_to_bronze",
        "notebook_task": {"notebook_path": "/ETL/bronze_ingest"}
    },
    {
        "task_key": "transform_to_silver",
        "depends_on": [{"task_key": "ingest_to_bronze"}],
        "pipeline_task": {"pipeline_id": "silver-pipeline-id"}
    },
    {
        "task_key": "aggregate_to_gold",
        "depends_on": [{"task_key": "transform_to_silver"}],
        "pipeline_task": {"pipeline_id": "gold-pipeline-id"}
    },
    {
        "task_key": "refresh_reports",
        "depends_on": [{"task_key": "aggregate_to_gold"}],
        "sql_task": {"warehouse_id": "warehouse-id", "query": {...}}
    }
]
```

### Pattern 2: Fan-Out Processing

```python
tasks = [
    {"task_key": "prepare_data"},
    {"task_key": "process_region_us", "depends_on": [{"task_key": "prepare_data"}]},
    {"task_key": "process_region_eu", "depends_on": [{"task_key": "prepare_data"}]},
    {"task_key": "process_region_apac", "depends_on": [{"task_key": "prepare_data"}]},
    {
        "task_key": "consolidate",
        "depends_on": [
            {"task_key": "process_region_us"},
            {"task_key": "process_region_eu"},
            {"task_key": "process_region_apac"}
        ]
    }
]
```

### Pattern 3: ML Training Pipeline

```python
tasks = [
    {"task_key": "prepare_features"},
    {"task_key": "train_model", "depends_on": [{"task_key": "prepare_features"}]},
    {"task_key": "evaluate_model", "depends_on": [{"task_key": "train_model"}]},
    {"task_key": "register_model", "depends_on": [{"task_key": "evaluate_model"}]}
]
```

## Performance Optimization

### 1. **Use Pools for Faster Starts**

```python
{
    "instance_pool_id": "pool-id-here",
    "num_workers": 4
}
```

### 2. **Enable Autoscaling**

```python
{
    "autoscale": {
        "min_workers": 2,
        "max_workers": 10
    }
}
```

### 3. **Optimize Task Parallelism**

```python
{
    "max_concurrent_runs": 5  # Allow multiple job runs in parallel
}
```

## When to Use Lakeflow Jobs

### ✅ Best For:
* Scheduled ETL/ELT pipelines
* Multi-step data workflows
* ML training and inference pipelines
* Orchestrating Spark Declarative Pipelines
* Complex task dependencies
* Production data operations

### ⚠️ Consider Alternatives For:
* Ad-hoc analysis → Use notebooks interactively
* Real-time streaming → Use Spark Declarative Pipelines directly
* Simple file processing → Use Lakeflow Connect
* External orchestration → Integrate with Airflow, Prefect, etc.

## Resources

* **Documentation**: [Databricks Jobs](https://docs.databricks.com/workflows)
* **API Reference**: [Jobs API 2.1](https://docs.databricks.com/api/workspace/jobs)
* **Asset Bundles**: [Databricks Asset Bundles](https://docs.databricks.com/dev-tools/bundles)
* **Best Practices**: [Production Jobs Guide](https://docs.databricks.com/workflows/jobs/jobs-best-practices.html)

---

**Related Technologies:**
* **Lakeflow Spark Declarative Pipelines**: For data transformation workflows
* **Lakeflow Connect**: For managed data ingestion
* **Databricks Asset Bundles (DABs)**: For CI/CD and deployment
* **MLflow**: For ML lifecycle management

# Delta Lake: Unified Storage Layer for Data Lakes

## Overview

**Delta Lake** is an open-source storage framework that brings ACID transactions, scalable metadata handling, and unified streaming/batch data processing to cloud data lakes. It runs on top of existing data lakes (S3, ADLS, GCS) and is 100% compatible with Apache Spark.

### Why Delta Lake?

Traditional data lakes suffer from:
* **No ACID guarantees** - Concurrent writes can corrupt data
* **Poor performance** - Full table scans for every query
* **No schema enforcement** - Data quality issues
* **Complex pipeline maintenance** - Separate systems for batch and streaming
* **Limited data versioning** - No time travel or rollback

Delta Lake solves these problems while maintaining the flexibility and scale of data lakes.

## Core Features

### 1. **ACID Transactions**
Serializable isolation ensures data consistency even with concurrent reads and writes.

```python
# Multiple writers can safely write to the same table
# Delta Lake ensures consistency
df1.write.format("delta").mode("append").save("/data/events")
df2.write.format("delta").mode("append").save("/data/events")

# Readers always see consistent snapshots
df = spark.read.format("delta").load("/data/events")
```

### 2. **Time Travel (Data Versioning)**
Query previous versions of your data for audits, rollbacks, or reproducibility.

```python
# Read data as it existed yesterday
df = spark.read.format("delta") \
    .option("timestampAsOf", "2024-04-22") \
    .load("/data/events")

# Read data at version 5
df = spark.read.format("delta") \
    .option("versionAsOf", 5) \
    .load("/data/events")

# View table history
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/data/events")
display(delta_table.history())
```

### 3. **Schema Enforcement and Evolution**
Automatically validate schema on writes and evolve schema safely.

```python
# Schema enforcement: Write fails if schema doesn't match
df_wrong_schema.write.format("delta").mode("append").save("/data/events")
# ^ Raises AnalysisException

# Schema evolution: Add new columns automatically
df_new_columns.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/data/events")

# Explicit schema evolution
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/data/events")
delta_table.upgrade()  # Upgrade to latest protocol version
```

### 4. **Unified Batch and Streaming**
Same table can be read/written in both batch and streaming modes.

```python
# Batch write
df.write.format("delta").mode("overwrite").save("/data/events")

# Streaming write
df_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/checkpoints/events") \
    .start("/data/events")

# Batch read
df_batch = spark.read.format("delta").load("/data/events")

# Streaming read
df_stream = spark.readStream.format("delta").load("/data/events")
```

### 5. **Scalable Metadata Handling**
Efficiently handles tables with millions of files and partitions.

```python
# Delta Lake maintains a transaction log instead of listing files
# Queries are fast regardless of table size
df = spark.read.format("delta") \
    .load("/data/events") \
    .where("date = '2024-04-23'")  # Fast partition pruning
```

### 6. **Upserts and Deletes**
Update and delete data efficiently without full rewrites.

```python
from delta.tables import DeltaTable
from pyspark.sql.functions import col

delta_table = DeltaTable.forPath(spark, "/data/customers")

# DELETE operation
delta_table.delete("customer_id = 123")

# UPDATE operation
delta_table.update(
    condition="status = 'pending'",
    set={"status": "'active'", "updated_at": "current_timestamp()"}
)

# MERGE (UPSERT) operation
updates = spark.read.format("json").load("/data/updates")

delta_table.alias("target").merge(
    updates.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(set={
    "name": "source.name",
    "email": "source.email",
    "updated_at": "current_timestamp()"
}).whenNotMatchedInsert(values={
    "customer_id": "source.customer_id",
    "name": "source.name",
    "email": "source.email",
    "created_at": "current_timestamp()"
}).execute()
```

## Creating Delta Tables

### Method 1: Python API

```python
# Create from DataFrame
df = spark.read.json("/data/raw/events.json")

# Write as Delta table
df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("date") \
    .save("/data/events")

# Register as a table in Unity Catalog
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.bronze.events")
```

### Method 2: SQL

```sql
-- Create empty table with schema
CREATE TABLE main.bronze.events (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    timestamp TIMESTAMP,
    properties MAP<STRING, STRING>,
    date DATE
)
USING DELTA
PARTITIONED BY (date)
LOCATION '/data/events'
COMMENT 'Raw event data from application';

-- Create table from query (CTAS)
CREATE TABLE main.silver.active_users
USING DELTA
AS
SELECT 
    user_id,
    MAX(timestamp) as last_active,
    COUNT(*) as event_count
FROM main.bronze.events
WHERE date >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY user_id;
```

### Method 3: Clone Existing Table

```sql
-- Shallow clone (metadata only, instant)
CREATE TABLE main.dev.events_test
SHALLOW CLONE main.prod.events;

-- Deep clone (full copy)
CREATE TABLE main.staging.events_backup
DEEP CLONE main.prod.events;

-- Clone at specific version
CREATE TABLE main.dev.events_snapshot
SHALLOW CLONE main.prod.events VERSION AS OF 10;
```

## Common Operations

### Reading Delta Tables

```python
# By path
df = spark.read.format("delta").load("/data/events")

# By table name
df = spark.table("main.bronze.events")

# With time travel
df = spark.read.format("delta") \
    .option("timestampAsOf", "2024-04-20T10:00:00") \
    .load("/data/events")

# Specific columns and filters
df = spark.read.format("delta") \
    .load("/data/events") \
    .select("user_id", "event_type", "timestamp") \
    .where("date = '2024-04-23'")
```

### Writing Delta Tables

```python
# Overwrite entire table
df.write.format("delta").mode("overwrite").save("/data/events")

# Append new data
df.write.format("delta").mode("append").save("/data/events")

# Overwrite specific partitions
df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "date = '2024-04-23'") \
    .save("/data/events")

# With schema evolution
df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/data/events")
```

### Merge (Upsert) Operations

```python
from delta.tables import DeltaTable

# Load target table
target = DeltaTable.forName(spark, "main.silver.customers")

# Load source data
source = spark.read.format("json").load("/data/updates/customers")

# Perform merge
target.alias("t").merge(
    source.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    condition="s.updated_at > t.updated_at",
    set={
        "name": "s.name",
        "email": "s.email",
        "status": "s.status",
        "updated_at": "s.updated_at"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "status": "s.status",
        "created_at": "s.created_at",
        "updated_at": "s.updated_at"
    }
).whenNotMatchedBySource(
    condition="t.status = 'active'"
).update(
    set={"status": "'inactive'", "updated_at": "current_timestamp()"}
).execute()
```

### SQL Merge

```sql
MERGE INTO main.silver.customers AS target
USING updates.customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
    UPDATE SET
        target.name = source.name,
        target.email = source.email,
        target.updated_at = source.updated_at
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, created_at)
    VALUES (source.customer_id, source.name, source.email, current_timestamp());
```

## Performance Optimization

### 1. **OPTIMIZE (Compaction)**
Merge small files into larger ones for better read performance.

```sql
-- Compact all files
OPTIMIZE main.bronze.events;

-- Compact specific partition
OPTIMIZE main.bronze.events WHERE date = '2024-04-23';

-- With Z-Ordering for column locality
OPTIMIZE main.bronze.events
ZORDER BY (user_id, event_type);
```

```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "main.bronze.events")

# Compact files
delta_table.optimize().executeCompaction()

# Z-Order optimization
delta_table.optimize().executeZOrderBy("user_id", "event_type")
```

### 2. **VACUUM (Remove Old Files)**
Delete files no longer referenced by the table (older than retention period).

```sql
-- Remove files older than 7 days (default retention)
VACUUM main.bronze.events;

-- Custom retention period (hours)
VACUUM main.bronze.events RETAIN 168 HOURS;

-- Dry run to see what would be deleted
VACUUM main.bronze.events DRY RUN;
```

```python
delta_table = DeltaTable.forName(spark, "main.bronze.events")

# Vacuum with 7-day retention
delta_table.vacuum(168)  # hours
```

**⚠️ Warning**: VACUUM permanently deletes files. Time travel to versions before the retention period will fail.

### 3. **Partitioning**
Organize data by frequently filtered columns.

```python
# Partition by date for time-series data
df.write.format("delta") \
    .partitionBy("date") \
    .save("/data/events")

# Multi-level partitioning
df.write.format("delta") \
    .partitionBy("year", "month", "day") \
    .save("/data/events")
```

```sql
CREATE TABLE main.bronze.events (
    event_id STRING,
    user_id STRING,
    timestamp TIMESTAMP,
    date DATE
)
USING DELTA
PARTITIONED BY (date);
```

### 4. **Data Skipping (Auto)**
Delta Lake automatically collects min/max statistics for columns.

```python
# Delta automatically skips files based on filters
df = spark.read.format("delta") \
    .load("/data/events") \
    .where("timestamp >= '2024-04-20' AND user_id = '12345'")

# View statistics
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/data/events")
display(delta_table.detail())
```

### 5. **Deletion Vectors (Efficient Deletes/Updates)**
Avoid rewriting entire files for small updates.

```sql
-- Enable deletion vectors (DV)
ALTER TABLE main.bronze.events 
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');

-- Now deletes/updates only write small DV files
DELETE FROM main.bronze.events WHERE user_id = '12345';
```

### 6. **Liquid Clustering (Alternative to Partitioning)**
Automatic data organization without the overhead of partitions.

```sql
CREATE TABLE main.bronze.events (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY (user_id, event_type);

-- Data is automatically organized for optimal query performance
SELECT * FROM main.bronze.events WHERE user_id = '12345';
```

## Advanced Features

### Change Data Feed (CDF)
Track row-level changes for incremental processing.

```sql
-- Enable CDF
ALTER TABLE main.bronze.events
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Read changes since version 5
SELECT *
FROM table_changes('main.bronze.events', 5);

-- Read changes in time range
SELECT *
FROM table_changes('main.bronze.events', '2024-04-20', '2024-04-23');
```

```python
# Read CDF in Python
changes = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 5) \
    .table("main.bronze.events")

display(changes)
# Columns: _change_type (insert/update_preimage/update_postimage/delete)
```

### Identity Columns (Auto-Increment)

```sql
CREATE TABLE main.bronze.events (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    event_type STRING,
    timestamp TIMESTAMP
)
USING DELTA;

-- IDs are automatically assigned
INSERT INTO main.bronze.events (event_type, timestamp)
VALUES ('click', current_timestamp());
```

### Generated Columns

```sql
CREATE TABLE main.bronze.events (
    timestamp TIMESTAMP,
    date DATE GENERATED ALWAYS AS (CAST(timestamp AS DATE)),
    year INT GENERATED ALWAYS AS (YEAR(timestamp)),
    month INT GENERATED ALWAYS AS (MONTH(timestamp))
)
USING DELTA;

-- Generated columns are computed automatically on write
INSERT INTO main.bronze.events (timestamp)
VALUES ('2024-04-23T10:30:00');
```

### Constraints and Checks

```sql
-- Add NOT NULL constraint
ALTER TABLE main.bronze.events
ALTER COLUMN user_id SET NOT NULL;

-- Add CHECK constraint
ALTER TABLE main.bronze.events
ADD CONSTRAINT valid_amount CHECK (amount > 0);

-- Drop constraint
ALTER TABLE main.bronze.events
DROP CONSTRAINT valid_amount;
```

## Table Maintenance

### View Table History

```sql
DESCRIBE HISTORY main.bronze.events;
```

```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "main.bronze.events")
history = delta_table.history()
display(history)
```

### Restore Table to Previous Version

```sql
-- Restore to version 10
RESTORE TABLE main.bronze.events TO VERSION AS OF 10;

-- Restore to timestamp
RESTORE TABLE main.bronze.events TO TIMESTAMP AS OF '2024-04-20T10:00:00';
```

```python
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "main.bronze.events")

# Restore to version
delta_table.restoreToVersion(10)

# Restore to timestamp
delta_table.restoreToTimestamp("2024-04-20T10:00:00")
```

### Table Details and Statistics

```sql
DESCRIBE DETAIL main.bronze.events;
```

```python
delta_table = DeltaTable.forName(spark, "main.bronze.events")
details = delta_table.detail()
display(details)
# Shows: numFiles, sizeInBytes, partitionColumns, etc.
```

## Best Practices

### 1. **Partition Strategy**
* **Do partition** by date/time for time-series data
* **Don't partition** if you have < 1TB of data
* **Avoid** high-cardinality partitions (e.g., user_id)
* **Consider** Liquid Clustering instead of partitioning

```python
# Good: Date partitioning for large time-series data
df.write.format("delta").partitionBy("date").save("/data/events")

# Bad: Too many partitions
df.write.format("delta").partitionBy("user_id").save("/data/events")
```

### 2. **Regular Optimization**

```python
# Schedule OPTIMIZE and VACUUM regularly
# Example: Daily maintenance job

from delta.tables import DeltaTable

# Optimize tables
tables = ["main.bronze.events", "main.silver.users", "main.gold.metrics"]

for table_name in tables:
    delta_table = DeltaTable.forName(spark, table_name)
    
    # Compact small files
    delta_table.optimize().executeCompaction()
    
    # Z-Order if beneficial
    # delta_table.optimize().executeZOrderBy("user_id")
    
    # Vacuum old files (30-day retention)
    delta_table.vacuum(720)  # 30 days * 24 hours
```

### 3. **Schema Evolution**

```python
# Explicit schema evolution
df_new_schema.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/data/events")

# Or set at table level
spark.sql("""
    ALTER TABLE main.bronze.events
    SET TBLPROPERTIES ('delta.autoMerge.mergeSchema' = 'true')
""")
```

### 4. **Idempotent Writes**

```python
# Use replaceWhere for idempotent partition overwrites
df.write.format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", "date = '2024-04-23'") \
    .save("/data/events")

# Or use MERGE for row-level idempotency
target.alias("t").merge(
    source.alias("s"),
    "t.event_id = s.event_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

### 5. **Monitor Table Growth**

```sql
-- Check table size and file count
DESCRIBE DETAIL main.bronze.events;

-- Identify tables needing optimization
SELECT
    name,
    num_files,
    size_in_bytes / 1024 / 1024 / 1024 as size_gb,
    num_files / (size_in_bytes / 1024 / 1024 / 1024) as files_per_gb
FROM (
    SELECT * FROM table_metadata
    WHERE num_files > 1000 OR size_in_bytes > 1000000000000
)
ORDER BY files_per_gb DESC;
```

## Common Patterns

### Pattern 1: Incremental ETL

```python
# Read new data since last checkpoint
last_processed_version = get_last_checkpoint("events_pipeline")

new_data = spark.read.format("delta") \
    .option("startingVersion", last_processed_version + 1) \
    .load("/data/events")

# Process and write
processed = transform(new_data)
processed.write.format("delta").mode("append").save("/data/processed_events")

# Update checkpoint
update_checkpoint("events_pipeline", current_version)
```

### Pattern 2: Slowly Changing Dimension (SCD Type 2)

```python
from pyspark.sql.functions import col, current_timestamp, lit

target = DeltaTable.forName(spark, "main.silver.customers_scd")

target.alias("t").merge(
    source.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.name != s.name OR t.email != s.email",
    set={
        "is_current": "false",
        "end_date": "current_timestamp()"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "is_current": "true",
        "start_date": "current_timestamp()",
        "end_date": "cast(null as timestamp)"
    }
).execute()

# Insert new versions for changed records
changed = source.join(
    target.toDF().where("is_current = false"),
    "customer_id"
)
changed.select(
    col("customer_id"),
    col("name"),
    col("email"),
    lit(True).alias("is_current"),
    current_timestamp().alias("start_date"),
    lit(None).alias("end_date")
).write.format("delta").mode("append").saveAsTable("main.silver.customers_scd")
```

### Pattern 3: Deduplication

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Read data with duplicates
df = spark.read.format("delta").load("/data/events")

# Deduplicate by keeping latest record per ID
window = Window.partitionBy("event_id").orderBy(col("timestamp").desc())

deduped = df.withColumn("row_num", row_number().over(window)) \
    .where("row_num = 1") \
    .drop("row_num")

# Write back
deduped.write.format("delta").mode("overwrite").save("/data/events_deduped")
```

## When to Use Delta Lake

### ✅ Best For:
* **Data lakes** requiring ACID guarantees
* **Streaming + batch** unified pipelines
* **Time-series data** with historical queries
* **Incremental ETL** pipelines
* **Slowly changing dimensions**
* **Audit and compliance** (time travel)
* **Lakehouse architectures**

### ⚠️ Consider Alternatives For:
* **Transactional databases** → Use operational databases (MySQL, PostgreSQL)
* **Real-time OLTP** → Use purpose-built OLTP systems
* **Extremely small datasets** → Parquet or CSV may suffice
* **Write-once, read-many archives** → Consider cheaper storage (Parquet on S3 Glacier)

## Performance Comparison

| Operation | Parquet | Delta Lake |
|-----------|---------|------------|
| Read (filtered) | Slow (full scan) | Fast (data skipping) |
| Write (append) | Fast | Fast |
| Update/Delete | Very slow (rewrite) | Fast (merge) |
| Concurrent writes | ❌ Not safe | ✅ ACID safe |
| Time travel | ❌ Not supported | ✅ Supported |
| Schema evolution | Manual | Automatic |
| Small files | Performance issues | Auto-compaction |

## Resources

* **Official Docs**: [Delta Lake Documentation](https://docs.delta.io)
* **Databricks Docs**: [Delta Lake on Databricks](https://docs.databricks.com/delta)
* **GitHub**: [Delta Lake GitHub](https://github.com/delta-io/delta)
* **Protocol**: [Delta Transaction Log Protocol](https://github.com/delta-io/delta/blob/master/PROTOCOL.md)

---

**Related Technologies:**
* **Unity Catalog**: Governance layer for Delta tables
* **Lakeflow Spark Declarative Pipelines**: Build Delta pipelines declaratively
* **Apache Spark**: Processing engine for Delta Lake
* **Apache Parquet**: Underlying file format

# Unity Catalog: Unified Data Governance

## Overview

**Unity Catalog** is Databricks' unified governance solution for data, AI, and analytics assets. It provides centralized access control, auditing, lineage, and data discovery across clouds and workspace boundaries.

### Why Unity Catalog?

Traditional data governance challenges:
* **Fragmented security** - Different access controls for tables, files, models, notebooks
* **No cross-cloud governance** - Separate systems for AWS, Azure, GCP
* **Limited lineage** - Can't trace data from source to consumption
* **Poor discoverability** - Hard to find and understand data assets
* **Compliance gaps** - Difficult to audit who accessed what data
* **Workspace silos** - Tables can't be shared across workspaces

Unity Catalog solves these with a single governance layer for all data assets.

---

## Core Concepts

### 1. **Three-Level Namespace**

Unity Catalog organizes data in a three-level hierarchy:

```
catalog.schema.object
```

**Example**: `main.sales.customers`
* **Catalog** (`main`) - Top-level container, typically per environment or business unit
* **Schema/Database** (`sales`) - Logical grouping of tables, views, functions
* **Object** (`customers`) - Tables, views, volumes, models, functions

```sql
-- Full object reference
SELECT * FROM main.sales.customers;

-- Set default catalog and schema
USE CATALOG main;
USE SCHEMA sales;

-- Now can use short names
SELECT * FROM customers;
```

### 2. **Securable Objects**

Unity Catalog governs these asset types:

| Object Type | Description | Example |
|-------------|-------------|----------|
| **Catalog** | Top-level container | `main`, `dev`, `prod` |
| **Schema** | Database/namespace | `sales`, `marketing`, `bronze` |
| **Table** | Structured data (managed or external) | `customers`, `orders` |
| **View** | Saved query | `active_customers` |
| **Volume** | Cloud storage location for files | `raw_data`, `models` |
| **Function** | User-defined function (UDF) | `calculate_discount()` |
| **Model** | MLflow registered model | `churn_predictor` |
| **Connection** | External database connection | `postgres_prod` |
| **Share** | Delta Sharing provider object | `customer_data_share` |

### 3. **Managed vs External Tables**

**Managed Tables:**
* Unity Catalog controls both metadata AND data
* Data stored in catalog's managed location
* Dropping table deletes data permanently

```sql
-- Create managed table
CREATE TABLE main.sales.customers (
    customer_id STRING,
    name STRING,
    email STRING
)
USING DELTA;

-- Data is stored in UC-managed location
-- DROP TABLE will delete data
```

**External Tables:**
* Unity Catalog controls metadata only
* Data stored in user-specified location
* Dropping table only removes metadata (data remains)

```sql
-- Create external table
CREATE TABLE main.sales.customers (
    customer_id STRING,
    name STRING,
    email STRING
)
USING DELTA
LOCATION 's3://my-bucket/customers/';

-- Data stays in your S3 bucket
-- DROP TABLE only removes metadata
```

### 4. **Storage Credentials & External Locations**

For external tables and volumes, Unity Catalog uses:

**Storage Credentials** - Authentication to cloud storage (IAM role, service principal)
**External Locations** - Specific cloud paths with associated credentials

```sql
-- Create storage credential (admin only)
CREATE STORAGE CREDENTIAL my_s3_cred
WITH (AWS_IAM_ROLE = 'arn:aws:iam::123456789:role/databricks-role');

-- Create external location
CREATE EXTERNAL LOCATION my_data_location
URL 's3://my-bucket/data/'
WITH (STORAGE CREDENTIAL my_s3_cred)
COMMENT 'Production data bucket';

-- Grant access to external location
GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION my_data_location TO `data_engineers`;
```

### 5. **Volumes** (Non-Tabular Data)

Volumes provide governed access to files (PDFs, images, models, etc.).

```sql
-- Create managed volume
CREATE VOLUME main.ml.model_artifacts
COMMENT 'MLflow model artifacts';

-- Create external volume
CREATE EXTERNAL VOLUME main.raw.incoming_files
LOCATION 's3://my-bucket/incoming/';

-- Grant access
GRANT READ VOLUME, WRITE VOLUME ON VOLUME main.ml.model_artifacts TO `ml_engineers`;
```

```python
# Access files in volumes
file_path = "/Volumes/main/ml/model_artifacts/model_v1.pkl"

import pickle
with open(file_path, 'rb') as f:
    model = pickle.load(f)

# List files in volume
import os
files = os.listdir("/Volumes/main/ml/model_artifacts/")
print(files)
```

---

## Access Control

### Privilege Model

Unity Catalog uses grant-based access control (not deny-based).

**Table-Level Privileges:**
* `SELECT` - Read table data
* `MODIFY` - Insert, update, delete data
* `READ_METADATA` - View table schema and metadata
* `CREATE TABLE` - Create tables in schema
* `ALL PRIVILEGES` - All permissions

**Schema-Level Privileges:**
* `USE SCHEMA` - Access the schema
* `CREATE TABLE`, `CREATE VIEW`, `CREATE FUNCTION`, `CREATE VOLUME`
* `ALL PRIVILEGES`

**Catalog-Level Privileges:**
* `USE CATALOG` - Access the catalog
* `CREATE SCHEMA` - Create schemas in catalog
* `ALL PRIVILEGES`

### Granting Permissions

```sql
-- Grant SELECT on table to group
GRANT SELECT ON TABLE main.sales.customers TO `analytics_team`;

-- Grant multiple privileges
GRANT SELECT, MODIFY ON TABLE main.sales.orders TO `data_engineers`;

-- Grant at schema level (applies to all current and future tables)
GRANT SELECT ON SCHEMA main.sales TO `business_users`;

-- Grant at catalog level
GRANT USE CATALOG ON CATALOG main TO `all_users`;
GRANT USE SCHEMA ON SCHEMA main.sales TO `all_users`;

-- Grant to user (instead of group)
GRANT ALL PRIVILEGES ON TABLE main.dev.test_table TO `user@example.com`;

-- Grant with cascade
GRANT USE CATALOG, USE SCHEMA, SELECT ON SCHEMA main.sales TO `analysts`;
```

### Revoking Permissions

```sql
-- Revoke specific privilege
REVOKE SELECT ON TABLE main.sales.customers FROM `contractors`;

-- Revoke all privileges
REVOKE ALL PRIVILEGES ON TABLE main.sales.orders FROM `old_team`;
```

### Checking Permissions

```sql
-- Show grants on object
SHOW GRANTS ON TABLE main.sales.customers;

-- Show grants to principal
SHOW GRANTS TO `analytics_team`;

-- Show grants on schema
SHOW GRANTS ON SCHEMA main.sales;

-- Show effective permissions (includes inherited)
SHOW EFFECTIVE GRANTS ON TABLE main.sales.customers;
```

### Ownership

Every securable object has an owner (user or group) with full control.

```sql
-- Transfer ownership
ALTER TABLE main.sales.customers OWNER TO `data_engineering`;

-- Transfer schema ownership
ALTER SCHEMA main.sales OWNER TO `sales_team`;

-- Transfer catalog ownership
ALTER CATALOG main OWNER TO `platform_team`;
```

---

## Row-Level and Column-Level Security

### Row Filters (Row-Level Security)

Restrict which rows users can see.

```sql
-- Create row filter function
CREATE FUNCTION main.security.customer_filter(region STRING)
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('us_analysts'), region = 'US', TRUE);

-- Apply to table
ALTER TABLE main.sales.customers
SET ROW FILTER main.security.customer_filter ON (region);

-- Now US analysts only see US customers
SELECT * FROM main.sales.customers;  -- Auto-filtered
```

### Column Masks (Column-Level Security)

Mask sensitive column values.

```sql
-- Create masking function
CREATE FUNCTION main.security.email_mask(email STRING)
RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('pii_access') THEN email
    ELSE CONCAT(SUBSTRING(email, 1, 2), '***@***.com')
END;

-- Apply to column
ALTER TABLE main.sales.customers
ALTER COLUMN email SET MASK main.security.email_mask;

-- Users without pii_access see masked emails
SELECT email FROM main.sales.customers;
-- Result: 'jo***@***.com' (instead of 'john@example.com')
```

### Dynamic Views (Alternative Approach)

Use views with access control logic.

```python
# Create dynamic view
spark.sql("""
    CREATE OR REPLACE VIEW main.sales.customers_filtered AS
    SELECT 
        customer_id,
        name,
        CASE 
            WHEN IS_ACCOUNT_GROUP_MEMBER('pii_access') THEN email
            ELSE 'REDACTED'
        END as email,
        region
    FROM main.sales.customers
    WHERE 
        CASE
            WHEN IS_ACCOUNT_GROUP_MEMBER('us_analysts') THEN region = 'US'
            WHEN IS_ACCOUNT_GROUP_MEMBER('eu_analysts') THEN region = 'EU'
            ELSE TRUE  -- Admins see all
        END
""")

# Grant access to view (not underlying table)
spark.sql("GRANT SELECT ON VIEW main.sales.customers_filtered TO `all_analysts`")
```

---

## Data Lineage

Unity Catalog automatically captures lineage for:
* Table-to-table dependencies
* Column-level lineage
* Notebook and query usage
* Dashboard dependencies

```sql
-- View lineage in Catalog Explorer UI
-- Or query lineage via system tables

SELECT 
    source_table,
    target_table,
    notebook_path,
    created_at
FROM system.access.table_lineage
WHERE target_table_full_name = 'main.gold.customer_metrics'
ORDER BY created_at DESC;
```

```python
# Python API for lineage
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get lineage for table
lineage = w.lineage.get_lineage(
    table_name="main.gold.customer_metrics",
    include_entity_lineage=True
)

print(f"Upstream tables: {lineage.upstreams}")
print(f"Downstream tables: {lineage.downstreams}")
```

---

## Auditing

Unity Catalog logs all access and changes.

```sql
-- Query audit logs via system tables
SELECT 
    event_time,
    user_identity.email as user_email,
    action_name,
    request_params.full_name_arg as table_name,
    response.result as status
FROM system.access.audit
WHERE action_name = 'getTable'
    AND event_date >= CURRENT_DATE - INTERVAL 7 DAYS
    AND request_params.full_name_arg LIKE 'main.sales.%'
ORDER BY event_time DESC
LIMIT 100;

-- Track who queried sensitive tables
SELECT 
    user_identity.email,
    COUNT(*) as query_count,
    MAX(event_time) as last_access
FROM system.access.audit
WHERE action_name = 'generateTemporaryTableCredential'
    AND request_params.table_id IN (
        SELECT table_id FROM system.information_schema.tables
        WHERE table_schema = 'pii_data'
    )
GROUP BY user_identity.email
ORDER BY query_count DESC;
```

---

## Common Operations

### Creating Catalogs and Schemas

```sql
-- Create catalog
CREATE CATALOG IF NOT EXISTS production
COMMENT 'Production data catalog';

-- Create schema
CREATE SCHEMA IF NOT EXISTS production.sales
COMMENT 'Sales department data'
LOCATION 's3://my-bucket/sales/';  -- Optional for external

-- Create schema with managed location
CREATE SCHEMA production.analytics;
```

```python
# Python API
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Create catalog
spark.sql("CREATE CATALOG IF NOT EXISTS dev")

# Create schema
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS dev.experiments
    COMMENT 'Data science experiments'
""")
```

### Creating Tables

```sql
-- Managed table (UC controls data)
CREATE TABLE main.sales.transactions (
    transaction_id STRING,
    customer_id STRING,
    amount DECIMAL(10,2),
    transaction_date DATE
)
USING DELTA
PARTITIONED BY (transaction_date)
COMMENT 'Daily transaction records';

-- External table (you control data)
CREATE EXTERNAL TABLE main.sales.legacy_transactions (
    transaction_id STRING,
    amount DECIMAL(10,2)
)
USING DELTA
LOCATION 's3://my-bucket/legacy/transactions/';

-- Create table from query (CTAS)
CREATE TABLE main.sales.monthly_summary
USING DELTA
AS
SELECT 
    DATE_TRUNC('month', transaction_date) as month,
    SUM(amount) as total_revenue,
    COUNT(*) as transaction_count
FROM main.sales.transactions
GROUP BY DATE_TRUNC('month', transaction_date);
```

```python
# Python API
df = spark.read.json("/data/raw/events.json")

# Write as managed table
df.write.format("delta") \
    .mode("overwrite") \
    .option("comment", "Raw event data") \
    .saveAsTable("main.bronze.events")

# Write as external table
df.write.format("delta") \
    .mode("overwrite") \
    .option("path", "s3://my-bucket/events/") \
    .saveAsTable("main.bronze.events_external")
```

### Working with Volumes

```sql
-- Create managed volume
CREATE VOLUME main.ml.model_files
COMMENT 'ML model artifacts and configs';

-- Create external volume
CREATE EXTERNAL VOLUME main.raw.incoming
LOCATION 's3://my-bucket/incoming/';

-- List volumes
SHOW VOLUMES IN main.ml;

-- Describe volume
DESCRIBE VOLUME EXTENDED main.ml.model_files;
```

```python
# Use volumes in Python
import os
import json

# Write file to volume
volume_path = "/Volumes/main/ml/model_files/"
config = {"learning_rate": 0.01, "epochs": 100}

with open(f"{volume_path}config.json", "w") as f:
    json.dump(config, f)

# Read file from volume
with open(f"{volume_path}config.json", "r") as f:
    config = json.load(f)

# List files
files = os.listdir(volume_path)
print(files)

# Use with pandas
import pandas as pd
df = pd.read_csv(f"{volume_path}data.csv")
```

### Information Schema Queries

```sql
-- List all catalogs
SELECT * FROM system.information_schema.catalogs;

-- List schemas in catalog
SELECT * 
FROM system.information_schema.schemata
WHERE catalog_name = 'main';

-- List tables with metadata
SELECT 
    table_catalog,
    table_schema,
    table_name,
    table_type,
    created,
    comment
FROM system.information_schema.tables
WHERE table_schema = 'sales'
ORDER BY created DESC;

-- List columns
SELECT 
    table_name,
    column_name,
    data_type,
    is_nullable,
    comment
FROM system.information_schema.columns
WHERE table_schema = 'sales'
    AND table_name = 'customers'
ORDER BY ordinal_position;

-- Find tables by pattern
SELECT DISTINCT table_name
FROM system.information_schema.tables
WHERE table_name LIKE '%customer%'
    AND table_catalog = 'main';
```

### Table Properties and Comments

```sql
-- Add comment to table
COMMENT ON TABLE main.sales.customers IS 'Master customer dimension table';

-- Add column comment
ALTER TABLE main.sales.customers ALTER COLUMN email COMMENT 'Primary contact email';

-- Set table properties
ALTER TABLE main.sales.customers SET TBLPROPERTIES (
    'quality' = 'gold',
    'owner' = 'sales-team@company.com',
    'pii' = 'true'
);

-- View table properties
SHOW TBLPROPERTIES main.sales.customers;

-- Describe table with details
DESCRIBE TABLE EXTENDED main.sales.customers;
```

---

## Tagging and Classification

### Tags (Key-Value Metadata)

```sql
-- Add tags to table
ALTER TABLE main.sales.customers SET TAGS ('pii' = 'true', 'tier' = 'gold');

-- Add tags to schema
ALTER SCHEMA main.sales SET TAGS ('department' = 'sales', 'region' = 'us');

-- Query tables by tag
SELECT 
    table_catalog,
    table_schema,
    table_name,
    tag_name,
    tag_value
FROM system.information_schema.table_tags
WHERE tag_name = 'pii' AND tag_value = 'true';
```

### Governed Tags (Enterprise Feature)

Controlled vocabulary tags with access control.

```python
# Governed tags require CLI or SDK (not SQL)
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Create governed tag (metastore admin only)
tag = w.system_schemas.create(
    metastore_id="<metastore_id>",
    schema_name="system",
    name="sensitivity",
    description="Data sensitivity level",
    allowed_values=["public", "internal", "confidential", "restricted"]
)

# Apply governed tag to table
w.tables.update(
    full_name="main.sales.customers",
    tags={"sensitivity": "confidential"}
)
```

---

## Data Sharing (Delta Sharing)

Share live data with external organizations without copying.

### Creating a Share (Provider)

```sql
-- Create share
CREATE SHARE customer_analytics
COMMENT 'Customer data for partners';

-- Add table to share
ALTER SHARE customer_analytics
ADD TABLE main.sales.customer_metrics
COMMENT 'Aggregated customer metrics';

-- Add partitions (optional)
ALTER SHARE customer_analytics
ADD TABLE main.sales.transactions PARTITION (region = 'US');

-- Create recipient
CREATE RECIPIENT partner_company
COMMENT 'Partner XYZ Inc';

-- Grant share to recipient
GRANT SELECT ON SHARE customer_analytics TO RECIPIENT partner_company;
```

### Consuming a Share (Recipient)

```sql
-- List available shares
SHOW SHARES;

-- Create catalog from share
CREATE CATALOG partner_data
USING SHARE provider_name.customer_analytics;

-- Query shared data (read-only)
SELECT * FROM partner_data.default.customer_metrics;
```

---

## Metastore Administration

### Metastore (Top-Level Container)

One metastore per region, shared across workspaces.

```sql
-- View current metastore
SELECT current_metastore();

-- List all catalogs in metastore
SHOW CATALOGS;

-- Set default catalog for workspace
ALTER WORKSPACE SET DEFAULT CATALOG main;
```

```python
# Create metastore (account admin via SDK/CLI only)
from databricks.sdk import AccountClient

a = AccountClient()

metastore = a.metastores.create(
    name="prod-metastore",
    storage_root="s3://my-bucket/metastore/",
    region="us-west-2"
)

# Assign metastore to workspace
a.metastores.assign(
    workspace_id=123456789,
    metastore_id=metastore.metastore_id,
    default_catalog_name="main"
)
```

---

## Migration Patterns

### Migrating from Hive Metastore

```sql
-- Sync Hive table to Unity Catalog
CREATE TABLE main.bronze.legacy_customers
USING DELTA
AS SELECT * FROM hive_metastore.default.customers;

-- Or create external table pointing to same location
CREATE EXTERNAL TABLE main.bronze.customers_external
USING DELTA
LOCATION 'dbfs:/user/hive/warehouse/customers';

-- Upgrade workspace to use Unity Catalog
-- 1. Create metastore
-- 2. Assign to workspace
-- 3. Set workspace default catalog
-- 4. Migrate tables using sync or CTAS
```

```python
# Batch migration script
tables_to_migrate = [
    ("hive_metastore.default.customers", "main.bronze.customers"),
    ("hive_metastore.default.orders", "main.bronze.orders"),
    ("hive_metastore.sales.transactions", "main.bronze.transactions")
]

for source, target in tables_to_migrate:
    print(f"Migrating {source} -> {target}")
    
    # Read from Hive
    df = spark.table(source)
    
    # Write to Unity Catalog
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(target)
    
    # Grant permissions
    spark.sql(f"GRANT SELECT ON TABLE {target} TO `data_users`")
    
    print(f"✓ Migrated {source}")
```

---

## Best Practices

### 1. **Catalog Organization**

```
Organize by environment or business domain:

Option A: Environment-based
├── prod (production data)
│   ├── sales
│   ├── marketing
│   └── finance
├── staging (pre-prod testing)
│   └── ...
└── dev (development sandbox)
    └── ...

Option B: Domain-based
├── main (shared production)
│   ├── bronze (raw)
│   ├── silver (cleaned)
│   └── gold (aggregated)
├── sales_domain
│   └── ...
└── ml_models
    └── ...
```

### 2. **Access Control Strategy**

```sql
-- Grant at highest appropriate level
-- Users need USE on parent objects to access children

-- Minimum for read access:
GRANT USE CATALOG ON CATALOG main TO `analysts`;
GRANT USE SCHEMA ON SCHEMA main.sales TO `analysts`;
GRANT SELECT ON SCHEMA main.sales TO `analysts`;  -- All tables in schema

-- For write access:
GRANT MODIFY ON SCHEMA main.bronze TO `data_engineers`;

-- Don't grant ALL PRIVILEGES unless necessary
-- Use groups, not individual users
-- Assign owners to catalogs/schemas for delegation
```

### 3. **Tagging and Documentation**

```sql
-- Always add comments
CREATE TABLE main.sales.customers (...)
COMMENT 'Master customer dimension - updated nightly via ETL job';

COMMENT ON TABLE main.sales.customers IS 'Master customer dimension - updated nightly';

-- Add column-level comments
ALTER TABLE main.sales.customers 
ALTER COLUMN email COMMENT 'Primary contact email (PII)';

-- Use consistent tags
ALTER TABLE main.sales.customers SET TAGS (
    'pii' = 'true',
    'data_owner' = 'sales-team@company.com',
    'refresh_schedule' = 'daily',
    'quality_tier' = 'gold'
);
```

### 4. **Managed vs External Tables**

**Use Managed Tables when:**
* Unity Catalog should manage full lifecycle
* You want DROP TABLE to delete data
* Standard use case for new tables

**Use External Tables when:**
* Migrating existing data
* Data shared across multiple systems
* You need fine-grained control over storage
* Integrating with non-Databricks tools

### 5. **Volume Usage**

```python
# Use volumes for non-tabular data
volume_root = "/Volumes/main/ml/artifacts/"

# Store ML artifacts
mlflow.log_artifact("model.pkl", artifact_path=volume_root)

# Store config files
with open(f"{volume_root}/config.yaml", "w") as f:
    yaml.dump(config, f)

# Store raw files before parsing
dbutils.fs.cp(
    "s3://incoming/data.csv",
    f"{volume_root}/incoming/data.csv"
)
```

### 6. **Monitoring and Auditing**

```sql
-- Regular audit queries
-- Who accessed PII tables?
SELECT DISTINCT user_identity.email
FROM system.access.audit
WHERE event_date >= CURRENT_DATE - 7
    AND request_params.full_name_arg LIKE '%pii%';

-- Which tables are unused?
SELECT 
    t.table_catalog,
    t.table_schema,
    t.table_name,
    MAX(a.event_time) as last_access
FROM system.information_schema.tables t
LEFT JOIN system.access.audit a
    ON a.request_params.full_name_arg = CONCAT(t.table_catalog, '.', t.table_schema, '.', t.table_name)
WHERE t.table_catalog = 'main'
GROUP BY t.table_catalog, t.table_schema, t.table_name
HAVING last_access < CURRENT_DATE - 90 OR last_access IS NULL;
```

---

## Unity Catalog vs. Alternatives

| Feature | Unity Catalog | Hive Metastore | AWS Glue |
|---------|---------------|----------------|----------|
| **Cross-cloud** | ✅ Yes | ❌ No | ❌ AWS only |
| **Cross-workspace** | ✅ Yes | ❌ No | ✅ Yes |
| **Fine-grained ACLs** | ✅ Row/column | ❌ Table only | ❌ Table only |
| **Audit logging** | ✅ Built-in | ❌ Limited | ✅ Via CloudTrail |
| **Lineage** | ✅ Automatic | ❌ No | ❌ No |
| **ML models** | ✅ Yes | ❌ No | ❌ No |
| **Delta Sharing** | ✅ Yes | ❌ No | ❌ No |
| **Volumes (files)** | ✅ Yes | ❌ No | ❌ No |

---

## Common Issues and Solutions

### Issue: "Insufficient privileges"

```sql
-- Check your permissions
SHOW GRANTS ON TABLE main.sales.customers;
SHOW EFFECTIVE GRANTS ON TABLE main.sales.customers;

-- Check parent object access
SHOW GRANTS ON SCHEMA main.sales;
SHOW GRANTS ON CATALOG main;

-- Request access from owner
DESCRIBE TABLE EXTENDED main.sales.customers;  -- See owner
```

### Issue: Can't find table

```sql
-- Check you're in the right catalog/schema
SELECT current_catalog(), current_schema();

-- Switch context
USE CATALOG main;
USE SCHEMA sales;

-- Use fully qualified name
SELECT * FROM main.sales.customers;
```

### Issue: External table location access error

```sql
-- Check storage credential and external location
SHOW STORAGE CREDENTIALS;
SHOW EXTERNAL LOCATIONS;

-- Verify you have access
SHOW GRANTS ON EXTERNAL LOCATION my_data_location;
```

---

## Resources

* **Official Docs**: [Unity Catalog Documentation](https://docs.databricks.com/data-governance/unity-catalog/index.html)
* **Best Practices**: [Unity Catalog Best Practices](https://docs.databricks.com/data-governance/unity-catalog/best-practices.html)
* **Migration Guide**: [Migrate to Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/migrate.html)
* **Security Model**: [Unity Catalog Privileges](https://docs.databricks.com/data-governance/unity-catalog/manage-privileges/index.html)

---

**Related Concepts:**
* **Delta Lake**: Storage format for Unity Catalog tables
* **Databricks SQL**: Query engine with Unity Catalog integration
* **MLflow**: Model registry integrated with Unity Catalog
* **Delta Sharing**: External data sharing built on Unity Catalog